In [6]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

RAW_DIR = "../data/raw"

df_feats, _ = get_features(RAW_DIR)

print("Dataset shape:", df_feats.shape)

MIN_MINUTES = 100

# MISMO FILTRO que antes (2008–2024, mínimo de minutos)
df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008)
    & (df_feats["season_end_year"] <= 2024)
    & (df_feats["minutes_played"] >= MIN_MINUTES)
].copy()

print("ML dataset:", df_ml.shape)
df_ml.head()

Dataset shape: (76833, 79)
ML dataset: (23442, 79)


,player_id,minutes_played,goals,assists,yellow_cards,second_yellow_cards,direct_red_cards,penalty_goals,matches_played,clean_sheets,...,won_champions,team_ucl_strength,Titles,win_rate,goals_per_game,num_trophies,ballon_dor_winner,player_name,position,main_position
11,10,201.0,10.0,8,2,0,0,0,27,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
12,10,283.0,17.0,10,3,0,0,2,34,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
13,10,679.0,4.0,1,3,0,0,0,33,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
14,10,972.0,2.0,1,2,0,0,0,22,0,...,0.0,0.000000,0.0,0.0,0.00,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
15,10,190.0,12.0,7,1,0,0,0,27,0,...,0.0,0.014706,0.0,0.4,1.55,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack


In [7]:
# ===  FINAL FEATURES DE DIMENSIONALITY REDUCTION  ===
final_features = [
    # === Selected 30 ===
    'a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1',
    'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w',
    'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta',
    'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height',
    'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta',
    'age_norm', 'team_ucl_strength', 'age_penalty',
    'matches_played_z_delta', 'gc_per90_z_lag1',
    'minutes_played_z_delta', 'season_end_year',
    'Titles', 'num_trophies', 'won_champions',

    # === Metadata (not scaled) ===
    'player_id', 'player_name'
]

# Nos aseguramos de no tener duplicados
final_features = list(dict.fromkeys(final_features))

# Metadatos que NO deben entrar al modelo, solo para reportar
metadata_cols = [
    'player_id',
    'player_name',
    'season_end_year',
    'minutes_played',        # esta no venía en final_features, pero la queremos conservar
]

# Intersección con las columnas del df (por seguridad)
missing = [c for c in final_features if c not in df_ml.columns]
if missing:
    print("⚠️ WARNING - these final_features are missing in df_ml:", missing)

available = [c for c in final_features if c in df_ml.columns]

# Columnas que el modelo realmente va a usar
model_feature_cols = [
    c for c in available
    if c not in metadata_cols  # sacamos metadatos
]

print("Num model features:", len(model_feature_cols))
print(model_feature_cols)


Num model features: 28
['a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1', 'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w', 'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta', 'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height', 'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta', 'age_norm', 'team_ucl_strength', 'age_penalty', 'matches_played_z_delta', 'gc_per90_z_lag1', 'minutes_played_z_delta', 'Titles', 'num_trophies', 'won_champions']


In [9]:
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()
df_test  = df_ml[df_ml["season_end_year"] >= 2023].copy()

print("\nTrain:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)



Train: (14301, 79)
Val:   (6038, 79)
Test:  (3103, 79)


In [10]:
X_train = df_train[model_feature_cols].copy()
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[model_feature_cols].copy()
y_val = df_val["ballon_dor_winner"].astype(int)

X_test = df_test[model_feature_cols].copy()
y_test = df_test["ballon_dor_winner"].astype(int)

print("\nTrain X:", X_train.shape)
print("Val X:", X_val.shape)
print("Test X:", X_test.shape)




Train X: (14301, 28)
Val X: (6038, 28)
Test X: (3103, 28)
